# Live tables

A random-walk ticker appends trades in the background; the session runs a cycle as soon as trades arrive, and each displayed table re-renders when a cycle changes it.

In [ ]:
import random
import threading
import time

import pyarrow as pa
import ursid as us

s = us.local()
ticks = s.input_table(
    {
        "sym": us.string(),
        "price": us.float64(),
        "size": us.int64(),
        "ts": us.timestamp(),
    }
)
stats = ticks.agg_by(
    "sym",
    n="count(*)",
    avg="mean(price)",
    vol="std(price)",
    low="min(price)",
    high="max(price)",
)
latest = ticks.where("size > 100").update("notional = price * size").last_by("sym")

In [ ]:
SYMBOLS = ["AAPL", "MSFT", "NVDA", "AMZN", "GOOG", "META", "TSLA", "AVGO", "JPM", "V"]
prices = dict.fromkeys(SYMBOLS, 100.0)
producing = threading.Event()


def produce() -> None:
    """Appends 200 trades every 50 ms until `producing` is cleared."""
    while producing.is_set():
        syms = random.choices(SYMBOLS, k=200)
        price = []
        for sym in syms:
            prices[sym] = max(0.01, prices[sym] + random.gauss(0, 0.05))
            price.append(round(prices[sym], 2))
        now = time.time_ns()
        ticks.append(
            pa.table(
                {
                    "sym": syms,
                    "price": price,
                    "size": [random.randint(1, 500) for _ in syms],
                    "ts": pa.array([now] * len(syms), pa.timestamp("ns", tz="UTC")),
                }
            )
        )
        time.sleep(0.05)


producing.set()
threading.Thread(target=produce, daemon=True).start()

In [ ]:
stats

In [ ]:
latest

In [ ]:
producing.clear()  # stop the ticker; `s.stop()` stops the cycles